# MarketPulse: 04 - Global Machine Learning Forecasting & Reduction

This research notebook explores **Global vs. Local Machine Learning Forecasting** using `sktime`'s reduction framework.

### Research Questions:
1. **Local vs. Global**: Does a shared model trained across multiple series outperform independently fitted local models?
2. **Recursive vs. Direct Reduction**: Does iterative multi-step forecasting accumulate error faster than direct horizon-specific regressors?
3. **Model Comparison**: How do tree-based gradient boosting models (`LightGBM`, `HistGradientBoosting`) compare to `RandomForest` and statistical baselines (`Theta`, `ARIMA`)?

### Strict Leakage Prevention:
- Lags $k \ge 1$: Features strictly precede target observation $y_t$.
- Rolling statistics: Computed on `shift(1)` so observation $y_t$ is never included in its own predictors.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import load_config
from src.data.hierarchy import M5Hierarchy
from src.features.lag_features import create_lag_features
from src.features.rolling_features import create_rolling_features
from src.forecasting.global_models import build_ml_reduction_models, create_reduction_forecaster
from src.evaluation.benchmarking import BaselineBenchmarkEngine

sns.set_theme(style="whitegrid", palette="muted")
config = load_config(project_root / "configs" / "config.yaml")
df = pd.read_parquet(config.get_processed_path(config.data.processed_file))
print(f"Loaded {len(df):,} records.")

## 1. Feature Engineering with Strict Shift (Zero Target Leakage)

We generate autoregressive lags (1, 7, 14, 28) and rolling statistics (7-day mean, 28-day std).

In [ ]:
# Demonstrate lag and rolling feature extraction on sample series
sample_series_df = df[df["id"] == df["id"].iloc[0]].sort_values("date").copy()
sample_series_df = create_lag_features(sample_series_df, target_col="sales", lags=[1, 7, 14, 28])
sample_series_df = create_rolling_features(sample_series_df, target_col="sales", mean_windows=[7, 28], std_windows=[7, 28], shift=1)

feature_cols = [c for c in sample_series_df.columns if c.startswith(("lag_", "rolling_"))]
print(f"Generated {len(feature_cols)} features:", feature_cols)
display(sample_series_df[["date", "sales"] + feature_cols].dropna().head(5))

## 2. Comparing Recursive vs. Direct Multi-Step Reduction with LightGBM

- **Recursive**: Trains a 1-step ahead model $f(y_{t-1}, \dots, y_{t-w}) \to \hat{y}_t$. Predictions are rolled into the input window for step $t+1$.
- **Direct**: Trains separate models $f_h(y_{t-1}, \dots, y_{t-w}) \to \hat{y}_{t+h}$ for each horizon $h$.

In [ ]:
hierarchy = M5Hierarchy(metadata_df=df)
total_demand = hierarchy.aggregate_bottom_up(df, level=1)
y_total = total_demand.set_index("date")["sales"].asfreq("D").fillna(0.0)

horizon = 28
y_train = y_total.iloc[:-horizon]
y_test = y_total.iloc[-horizon:]

lgb_rec = create_reduction_forecaster("lightgbm", strategy="recursive", window_length=14)
lgb_dir = create_reduction_forecaster("lightgbm", strategy="direct", window_length=14)

res_rec = lgb_rec.fit_and_evaluate(y_train=y_train, y_test=y_test, series_id="Total_Demand")
res_dir = lgb_dir.fit_and_evaluate(y_train=y_train, y_test=y_test, series_id="Total_Demand")

print(f"LightGBM Recursive | WAPE: {res_rec.metrics['WAPE']:.2f}% | RMSE: {res_rec.metrics['RMSE']:.2f} | Fit: {res_rec.fit_time_sec:.4f}s")
print(f"LightGBM Direct    | WAPE: {res_dir.metrics['WAPE']:.2f}% | RMSE: {res_dir.metrics['RMSE']:.2f} | Fit: {res_dir.fit_time_sec:.4f}s")

plt.figure(figsize=(14, 5))
plt.plot(y_test.index, y_test.values, label="Actual Demand", color="black", lw=2)
plt.plot(y_test.index, res_rec.predictions, label=f"LightGBM Recursive (WAPE {res_rec.metrics['WAPE']:.1f}%)", color="#2980b9", lw=2)
plt.plot(y_test.index, res_dir.predictions, label=f"LightGBM Direct (WAPE {res_dir.metrics['WAPE']:.1f}%)", color="#e74c3c", linestyle="--", lw=2)
plt.title("Recursive vs. Direct Multi-Step Forecasting (28 Days)")
plt.xlabel("Date")
plt.ylabel("Units")
plt.legend()
plt.show()

## 3. Benchmark Comparison Across Regressors (LightGBM vs HistGB vs Random Forest)

In [ ]:
models = build_ml_reduction_models(window_length=14)
engine = BaselineBenchmarkEngine(forecasters=models, horizons=[7, 14, 28], fixed_origin=True)
engine.evaluate_series(y_total, series_id="Total_Demand")
leaderboard = engine.get_leaderboard(sort_by="WAPE")
display(leaderboard)